In [1]:
import pandas as pd
import plotly.express as px

# Load the correct sheet
df = pd.read_excel("GuttmacherInstituteAbortionDataByState.xlsx", sheet_name="Guttmacher")

# -----------------------------
# Party / political lean mapping
# -----------------------------
democrat_states = [
    "California", "Colorado", "Connecticut", "Delaware", "Hawaii",
    "Illinois", "Maine", "Maryland", "Massachusetts", "Minnesota",
    "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York",
    "Oregon", "Rhode Island", "Vermont", "Virginia", "Washington",
    "District of Columbia"
]

republican_states = [
    "Alabama", "Alaska", "Arkansas", "Florida", "Idaho", "Indiana",
    "Iowa", "Kansas", "Kentucky", "Louisiana", "Mississippi", "Missouri",
    "Montana", "Nebraska", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "South Carolina", "South Dakota", "Tennessee", "Texas",
    "Utah", "West Virginia", "Wyoming"
]

swing_states = [
    "Arizona", "Georgia", "Michigan", "Pennsylvania", "Wisconsin"
]

party_map = (
    {s: "Democrat" for s in democrat_states} |
    {s: "Republican" for s in republican_states} |
    {s: "Swing" for s in swing_states}
)

df["political_lean"] = df["U.S. State"].map(party_map)

color_map = {
    "Democrat": "#1f77b4",
    "Republican": "#d62728",
    "Swing": "#9467bd"
}

# -----------------------------
# Useful derived columns
# -----------------------------
df["net_flow"] = (
    df["No. of abortions, by state of occurrence, 2020"]
    - df["No. of abortions, by state of residence, 2020"]
)

# make sure numeric columns are actually numeric
numeric_cols = [
    "% of residents obtaining abortions who traveled out of state for care, 2020",
    "% of women aged 15-44 living in a county without a clinic, 2020",
    "No. of abortion clinics, 2020",
    "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020",
    "% change in the no. of abortion clinics, 2017-2020",
    "% change in abortion rate, 2017-2020",
    "Total reported public expenditures for abortions (in 000s of dollars), 2015",
    "Total no. of publicly funded abortions , 2010",
    "No. of abortions, by state of occurrence, 2020",
    "No. of abortions, by state of residence, 2020"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [2]:
df_sorted = df.sort_values("net_flow", ascending=True)

fig = px.bar(
    df_sorted,
    x="U.S. State",
    y="net_flow",
    color="political_lean",
    color_discrete_map=color_map,
    title="Net Abortion Flow by State (2020)",
    labels={
        "U.S. State": "State",
        "net_flow": "Net Flow (Occurrence - Residence)",
        "political_lean": "Political Lean"
    },
    hover_name="U.S. State"
)

fig.update_layout(
    height=700,
    xaxis=dict(tickangle=45)
)

fig.show()

In [3]:
col_x = "No. of abortion clinics, 2020"
col_y = "% of residents obtaining abortions who traveled out of state for care, 2020"

fig = px.scatter(
    df,
    x=col_x,
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Number of Clinics vs Out-of-State Travel",
    labels={
        col_x: "Number of Abortion Clinics (2020)",
        col_y: "% of Residents Traveling Out of State (2020)",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [4]:
col_x = "% of women aged 15-44 living in a county without a clinic, 2020"
col_y = "% of residents obtaining abortions who traveled out of state for care, 2020"

fig = px.scatter(
    df,
    x=col_x,
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Clinic Deserts vs Out-of-State Travel",
    labels={
        col_x: "% Women Living in Counties Without a Clinic (2020)",
        col_y: "% of Residents Traveling Out of State (2020)",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [5]:
exporters = df.sort_values("net_flow").head(15)

fig = px.bar(
    exporters,
    x="U.S. State",
    y="net_flow",
    color="political_lean",
    color_discrete_map=color_map,
    title="Top Net Exporter States",
    labels={
        "U.S. State": "State",
        "net_flow": "Net Flow (Occurrence - Residence)",
        "political_lean": "Political Lean"
    }
)

fig.update_layout(xaxis=dict(tickangle=45))
fig.show()

In [6]:
importers = df.sort_values("net_flow", ascending=False).head(15)

fig = px.bar(
    importers,
    x="U.S. State",
    y="net_flow",
    color="political_lean",
    color_discrete_map=color_map,
    title="Top Net Importer States",
    labels={
        "U.S. State": "State",
        "net_flow": "Net Flow (Occurrence - Residence)",
        "political_lean": "Political Lean"
    }
)

fig.update_layout(xaxis=dict(tickangle=45))
fig.show()

## Anti-Access

In [7]:
col_y = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

fig = px.box(
    df,
    x="political_lean",
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    points="all",
    category_orders={"political_lean": ["Democrat", "Swing", "Republican"]},
    hover_name="U.S. State",
    title="Abortion Rate by Political Lean",
    labels={
        "political_lean": "Political Lean",
        col_y: "Abortions per 1,000 Women Aged 15–44"
    }
)

fig.show()

In [8]:
col_x = "No. of abortion clinics, 2020"
col_y = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

fig = px.scatter(
    df,
    x=col_x,
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Number of Clinics vs Abortion Rate",
    labels={
        col_x: "Number of Abortion Clinics (2020)",
        col_y: "Abortions per 1,000 Women Aged 15–44",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [15]:
df["clinic_group"] = pd.cut(
    df["No. of abortion clinics, 2020"],
    bins=[0, 10, 20, 100],
    labels=["Low (0–10)", "Medium (10–20)", "High (20+)"]
)

grouped = df.groupby("clinic_group")[
    "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"
].mean().reset_index()

fig = px.bar(
    grouped,
    x="clinic_group",
    y="No. of abortions per 1,000 women aged 15–44, by state of residence, 2020",
    title="States with More Clinics Have Higher Average Abortion Rates",
    labels={
        "clinic_group": "Number of Clinics",
        "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020":
        "Avg Abortions per 1,000 Women"
    }
)

fig.show()

/var/folders/_p/jpt6f06s11j09v6s60d6ps2c0000gn/T/ipykernel_25671/2722362754.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby("clinic_group")[


In [9]:
col_x = "% change in the no. of abortion clinics, 2017-2020"
col_y = "% change in abortion rate, 2017-2020"

fig = px.scatter(
    df,
    x=col_x,
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Change in Clinics vs Change in Abortion Rate",
    labels={
        col_x: "% Change in Number of Clinics (2017-2020)",
        col_y: "% Change in Abortion Rate (2017-2020)",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [16]:
col_x = "% of residents obtaining abortions who traveled out of state for care, 2020"
col_y = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

fig = px.scatter(
    df,
    x=col_x,
    y=col_y,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Out-of-State Travel vs Abortion Rate",
    labels={
        col_x: "% of Residents Traveling Out of State (2020)",
        col_y: "Abortions per 1,000 Women Aged 15–44",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [18]:
import pandas as pd
import plotly.express as px

col_x = "% of residents obtaining abortions who traveled out of state for care, 2020"
col_y = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

# Keep only Democrat and Republican states
df_persuade = df[df["political_lean"].isin(["Democrat", "Republican"])].copy()

# Remove a few extreme high-rate Democratic outliers that stretch the y-axis
df_persuade = df_persuade[df_persuade[col_y] <= 20].copy()

# Optional: remove very high travel outliers if you want an even tighter visual
# df_persuade = df_persuade[df_persuade[col_x] <= 90].copy()

fig = px.scatter(
    df_persuade,
    x=col_x,
    y=col_y,
    color="political_lean",
    trendline="ols",
    hover_name="U.S. State",
    color_discrete_map={
        "Republican": "#c62828",
        "Democrat": "#1565c0"
    },
    title="Even When More Residents Travel Out of State, Restrictive States Maintain Lower Abortion Rates",
    labels={
        col_x: "% of Residents Traveling Out of State (2020)",
        col_y: "Abortions per 1,000 Women Aged 15–44",
        "political_lean": "Political Lean"
    }
)

# Make the chart less visually messy / more persuasive
fig.update_traces(marker=dict(size=9, opacity=0.85))

fig.update_layout(
    width=1100,
    height=700,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=18),
    title=dict(x=0.5),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(t=120, l=80, r=40, b=80)
)

# Tighter axes so the difference is easier to see
fig.update_xaxes(
    range=[0, 100],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False
)

fig.update_yaxes(
    range=[0, 20],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False
)

fig.show()

In [11]:
df_occ = df.sort_values("No. of abortions, by state of occurrence, 2020", ascending=False)

fig = px.bar(
    df_occ,
    x="U.S. State",
    y="No. of abortions, by state of occurrence, 2020",
    color="political_lean",
    color_discrete_map=color_map,
    title="Total Abortions by State of Occurrence (2020)",
    labels={
        "U.S. State": "State",
        "No. of abortions, by state of occurrence, 2020": "Number of Abortions",
        "political_lean": "Political Lean"
    }
)

fig.update_layout(xaxis=dict(tickangle=45), height=700)
fig.show()

In [12]:
col_x = "% of residents obtaining abortions who traveled out of state for care, 2020"
col_y = "Total reported public expenditures for abortions (in 000s of dollars), 2015"
col_size = "Total no. of publicly funded abortions , 2010"

df_bubble = df.copy()
df_bubble[col_size] = df_bubble[col_size].fillna(0)
df_bubble["bubble_size"] = df_bubble[col_size].clip(lower=1)

fig = px.scatter(
    df_bubble,
    x=col_x,
    y=col_y,
    size="bubble_size",
    color="political_lean",
    color_discrete_map=color_map,
    hover_name="U.S. State",
    trendline="ols",
    size_max=40,
    title="Out-of-State Travel vs Public Abortion Funding",
    labels={
        col_x: "% of Residents Traveling Out of State (2020)",
        col_y: "Public Abortion Expenditures ($000s, 2015)",
        "bubble_size": "Publicly Funded Abortions",
        "political_lean": "Political Lean"
    }
)

fig.show()

In [13]:
candidate_pairs = [
    ("No. of abortion clinics, 2020",
     "% of residents obtaining abortions who traveled out of state for care, 2020"),
    
    ("% of women aged 15-44 living in a county without a clinic, 2020",
     "% of residents obtaining abortions who traveled out of state for care, 2020"),
    
    ("No. of abortion clinics, 2020",
     "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"),
    
    ("% change in the no. of abortion clinics, 2017-2020",
     "% change in abortion rate, 2017-2020"),
    
    ("% of residents obtaining abortions who traveled out of state for care, 2020",
     "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020")
]

for x_col, y_col in candidate_pairs:
    fig = px.scatter(
        df,
        x=x_col,
        y=y_col,
        color="political_lean",
        color_discrete_map=color_map,
        trendline="ols",
        hover_name="U.S. State",
        title=f"{x_col} vs {y_col}"
    )
    fig.show()